outputs/stream-14261982-1f8d-4821-9009-c35a65e55353/session_messages_data_20251106_201221.csv

In [12]:
from dotenv import load_dotenv

# Load environment variables from .env file
load_dotenv()

True

In [ ]:
import os 

from phoenix.client import Client
from phoenix.client.types.spans import SpanQuery
from datetime import datetime, timedelta


base_url = os.getenv("PHOENIX_BASE_URL")
api_key = os.getenv("PHOENIX_API_KEY")
client = Client(
            base_url=base_url,
            api_key=api_key
        )


attributes = [
        "span_kind",
        "span_id",
        "trace_id",
        "end_time",
        "session_id",
        "input.value",
        "output.value",
        "metadata"
    ]

query_end = (
    SpanQuery()
    .where("output.value == '__end__'")
    .select(*attributes)
)

# Optionally restrict to a time window to reduce data
start = datetime.now() - timedelta(days=3)

df = client.spans.get_spans_dataframe(query=query_end)

# Show first 5 rows
print(df.head())

# Show shape (rows, columns)
print(df.shape)

# Show column names
print(df.columns)

# Show data types
print(df.dtypes)

# Peek specific columns
print(df[["session_id", "output.value", "end_time"]].head())


                 span_kind  ...                                           metadata
context.span_id             ...                                                   
c6a304088e41662e     CHAIN  ...  {'thread_id': 'stream-e5347b84-f646-451c-b211-...
012b7558e1e0063e     CHAIN  ...  {'thread_id': 'stream-e5347b84-f646-451c-b211-...
457577c779cbe1cb     CHAIN  ...  {'thread_id': 'stream-e5347b84-f646-451c-b211-...
5753f639a3ef273f     CHAIN  ...  {'thread_id': 'stream-e5347b84-f646-451c-b211-...

[4 rows x 7 columns]
(4, 7)
Index(['span_kind', 'context.trace_id', 'end_time', 'session_id',
       'input.value', 'output.value', 'metadata'],
      dtype='object')
span_kind                        object
context.trace_id                 object
end_time            datetime64[ns, UTC]
session_id                       object
input.value                      object
output.value                     object
metadata                         object
dtype: object
                 session_id output.value

In [26]:

from phoenix.client import Client
from phoenix.client.types.spans import SpanQuery
import pandas as pd
from datetime import datetime, timedelta

# --- Client configuration ---
client = Client(
            base_url=base_url,
            api_key=api_key
        )

# --- Step 1: query all __end__ spans ---
attributes = [
    "span_kind",
    "span_id",
    "trace_id",
    "end_time",
    "session_id",
    "input.value",
    "output.value",
    "metadata"
]

query_end = (
    SpanQuery()
    .where("output.value == '__end__'")
    .select(*attributes)
)

df = client.spans.get_spans_dataframe(query=query_end)

# --- Step 2: pick the latest span per session ---
df["end_time"] = pd.to_datetime(df["end_time"], errors="coerce")
df = df.sort_values(["session_id", "end_time"])
latest = df.groupby("session_id").tail(1)

print(latest[["session_id", "trace_id", "span_id", "end_time", "output.value"]])

KeyError: "['trace_id', 'span_id'] not in index"

In [5]:
import pandas as pd

df_end["end_time"] = pd.to_datetime(df_end["end_time"])
df_sorted = df_end.sort_values(["session_id", "end_time"])
latest_per_session = df_sorted.groupby("session_id").tail(1)

KeyError: 'session_id'

In [ ]:
import json
import sys



def extract_conversation(json_data):
    """Extracts readable message interactions from the JSON conversation log."""
    messages = json_data.get("messages", [])
    output = []

    for msg in messages:
        role = msg.get("type")
        content = (msg.get("content") or "").strip()

        # Skip empty or tool-type messages
        if not content or role not in ("human", "ai"):
            continue

        role_name = "Human" if role == "human" else "AI"
        output.append(f"{role_name}:\n{content}\n")

    return "\n".join(output)


def main():
    """Reads JSON from stdin or a file path and prints extracted conversation."""
    if len(sys.argv) > 1:
        # Read from a file
        with open(sys.argv[1], "r", encoding="utf-8") as f:
            data = json.load(f)
    else:
        # Read from stdin
        data = json.load(sys.stdin)

    transcript = extract_conversation(data)
    print("\n--- Conversation Transcript ---\n")
    print(transcript)


if __name__ == "__main__":
    main()


Total sessions: 1 | Last traces: 1
Total extracted messages: 13
                         timestamp  \
0 2025-11-06 17:22:37.324624+00:00   
1 2025-11-06 17:22:37.325195+00:00   
2 2025-11-06 17:22:37.325195+00:00   
3 2025-11-06 17:22:37.325195+00:00   
4 2025-11-06 17:22:37.325195+00:00   
5 2025-11-06 17:22:40.800033+00:00   
6 2025-11-06 17:22:40.802717+00:00   
7 2025-11-06 17:22:40.917948+00:00   
8 2025-11-06 17:22:40.919882+00:00   
9 2025-11-06 17:22:40.920762+00:00   

                                    session_id  \
0  stream-14261982-1f8d-4821-9009-c35a65e55353   
1  stream-14261982-1f8d-4821-9009-c35a65e55353   
2  stream-14261982-1f8d-4821-9009-c35a65e55353   
3  stream-14261982-1f8d-4821-9009-c35a65e55353   
4  stream-14261982-1f8d-4821-9009-c35a65e55353   
5  stream-14261982-1f8d-4821-9009-c35a65e55353   
6  stream-14261982-1f8d-4821-9009-c35a65e55353   
7  stream-14261982-1f8d-4821-9009-c35a65e55353   
8  stream-14261982-1f8d-4821-9009-c35a65e55353   
9  stream-1426198

/tmp/ipykernel_2247847/2392657543.py:16: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: x.loc[x['end_time'].idxmax()])
